# Kimi-Linear (GDN-2) Code LM — train on **Google Colab (T4)**

Trains the hybrid Gated-DeltaNet-2 / MLA / MoE code language model on **CodeParrot**
using the `configs/colab_t4.yaml` recipe (~190M params, bfloat16, single GPU).

**Before you start:** `Runtime -> Change runtime type -> Hardware accelerator = T4 GPU`.

The pipeline: install -> get code -> prepare data (tokenize) -> train -> evaluate -> generate.


## 0. Confirm the GPU

In [ ]:
!nvidia-smi

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except:
    print("Not running on Google Colab, skipping drive mount.")

In [ ]:
try:
    from huggingface_hub import notebook_login
    notebook_login()
except:
    print("Cannot login to Hugging Face Hub.")

## 1. Get the project

Set `REPO_URL` to your repository, or upload the project folder to `/content` and
skip the clone. Everything below assumes we `cd` into the project root.


In [ ]:
import os

REPO_URL = "https://github.com/wisnunugroho21/nugie-codeparrot.git"  # <-- EDIT ME
PROJECT_DIR = "/content/nugie-codeparrot"

if not os.path.isdir(PROJECT_DIR):
    !git clone $REPO_URL $PROJECT_DIR
%cd $PROJECT_DIR
!ls


## 2. Install dependencies

We install the project's requirements first, then overlay the **CUDA 12 build of JAX**
last so the GPU plugin wins.


In [ ]:
!pip install -q -U -r requirements.txt
!pip install -q -U "jax[cuda12]"


## 3. (Optional) Hugging Face token

CodeParrot streams fine anonymously, but a token raises the download rate limit. Skip
this cell if you don't have one.


In [ ]:
# from huggingface_hub import login
# login()  # paste your HF token when prompted


## 4. Verify JAX sees the T4

In [ ]:
import jax
print("JAX", jax.__version__, "devices:", jax.devices())
assert any(d.platform == "gpu" for d in jax.devices()), "No GPU visible — set the runtime to T4."


## 5. Build the config

We load `configs/colab_t4.yaml` and apply **demo-friendly overrides** (small corpus,
few steps) so this notebook finishes quickly. Delete the overrides for a real run.
We reuse this one `cfg` object for every stage below.


In [ ]:
from pipeline.config import ExperimentConfig

cfg = ExperimentConfig.load("configs/colab_t4.yaml")

# --- Demo overrides (comment out for a full training run) ---
cfg.data.num_train_docs = 5000      # full config: 50000
cfg.data.num_val_docs   = 200
cfg.train.max_steps     = 2000      # full config: 40000
cfg.train.warmup_steps  = 100
cfg.train.eval_every    = 500
cfg.train.save_every    = 500
cfg.train.log_every     = 25

# --- Optional: persist checkpoints to Google Drive (Colab runtimes are ephemeral) ---
# from google.colab import drive; drive.mount("/content/drive")
# cfg.train.ckpt_dir = "/content/drive/MyDrive/nugie-codeparrot/checkpoints/colab_t4"

cfg.validate()
print("compute_dtype:", cfg.model.compute_dtype, "| seq_len:", cfg.data.seq_len,
      "| batch:", cfg.train.batch_size, "| max_steps:", cfg.train.max_steps)


## 6. Prepare the data

Streams `codeparrot/codeparrot-clean` from the Hub, tokenizes with the CodeParrot BPE
tokenizer, and writes packed `train.bin` / `val.bin` memmaps + `meta.json`. Runs once.


In [ ]:
from pipeline import prepare_data
prepare_data.prepare(cfg)


## 7. Train

Streams metrics (loss / perplexity / aux / lr / tokens-per-sec) and writes Orbax
checkpoints to `cfg.train.ckpt_dir`. Re-run with `train.train(cfg, resume=True)` to
continue from the latest checkpoint.


In [ ]:
from pipeline import train
train.train(cfg)


## 8. Evaluate

Restores the latest checkpoint and reports validation cross-entropy / perplexity /
bits-per-token over a capped number of batches.


In [ ]:
from pipeline import evaluate
evaluate.run_eval(cfg, step=None, max_batches=50)


## 9. Generate code

Autoregressive completion via the model's streaming decode. NOTE: after only the short
demo run above the output will be weak — train for many more steps for good code.


In [ ]:
evaluate.run_generate(cfg, step=None, prompt="def fibonacci(n):\n", max_new_tokens=128)
